## BUSCO output analysis
In this notebook, we will perform a comprehensive analysis based on the BUSCO output files from our dataset. The workflow is designed to move from basic quality control to intermediate comparative genomics, preparing our data for downstream evolutionary studies.

The analysis is divided into four main steps:

1. Quality Assessment Visualization: We will parse the BUSCO summary statistics for each proteome and generate a comparative stacked bar chart to visualize overall completeness across the dataset.

2. Quality Filtering: We will establish a basic filtering pipeline to identify and retain only the high-quality proteomes that pass a specific completeness threshold (e.g., $\ge80\%$).

3. Presence/Absence Heatmap: We will dive into the detailed gene tables (full_table.tsv) to create a presence/absence matrix. This allows us to visualize the specific status (Complete, Duplicated, Fragmented, Missing) of individual BUSCO genes across all species using a custom discrete heatmap.

4. Core Ortholog Extraction: Finally, we will identify and extract a list of universal single-copy orthologs—genes that are present in exactly one copy across all our analyzed species. These core genes serve as the perfect input for building species trees in subsequent modules.

In [17]:
# module initialization
import sys,os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

In [ ]:
# set paths
BUSCO_RES_FOLDER = '/vol/Topic4CommonData/Module1/precomputed_output/busco/'
MAPPING_PATH = '/vol/Topic4CommonData/Module1/proteome_list.txt'
OUT_PATH = '~/SIBBiodiversitySummerSchool2026Topic4/Module1_AnnotationQuality/out/busco/analysis/'
# TEST PATHS
BUSCO_RES_FOLDER = '/Users/spascare/data/work/dessimoz/teaching/2026/accessory_SIBBS/CommonDataModule1/precomputed_output/minidataset/busco/'
MAPPING_PATH = '/Users/spascare/data/work/dessimoz/teaching/2026/SIBBiodiversitySummerSchool2026Topic4/CommonData/proteome_list.txt'
OUT_PATH = '/Users/spascare/data/work/dessimoz/teaching/2026/SIBBiodiversitySummerSchool2026Topic4/Module1_AnnotationQuality/out/busco/analysis/'
if not os.path.exists(OUT_PATH):
    os.makedirs(OUT_PATH)

# Summary stacked bar plot
We will print the basic statistics for all the busco runs. Write here a plot that captures the basic BUSCO statistics found in the short summary file and plot it as an annotated bar plot. You will need to collect the statistics in each proteome output folder inside the file short_summary*.json. Then you will to plot it as a stacked bar graph (you can do this with the pandas dataframe utilities). bonus points if you use the same color map as the BUSCO plots.
In alternative, you can collect all *short_summary.specific* files in a folder and create the plot with `busco --plot DIRECTORY` instead (remember to mount the correct folder if you are using docker).

In [ ]:
summary_data = []
proteomes = [prot for prot in next(os.walk(BUSCO_RES_FOLDER))[1]]
for species in proteomes:
    species_dir = os.path.join(BUSCO_RES_FOLDER, species)
    # Search for the JSON summary file in the species directory
    json_files = glob.glob(os.path.join(species_dir, 'short_summary*.json'))
    if json_files:
        with open(json_files[0], 'r') as f:
            data = json.load(f)
            # Standard keys for BUSCO v5
            summary_data.append({
                'Species': species,
                'Complete_Single': data['results'].get('Single copy percentage', 0),
                'Complete_Duplicated': data['results'].get('Multi copy percentage', 0),
                'Fragmented': data['results'].get('Fragmented percentage', 0),
                'Missing': data['results'].get('Missing percentage', 0)
            })

# Create DataFrame
df_summary = pd.DataFrame(summary_data).set_index('Species')

# Plot the stacked bar chart
ax = df_summary.plot(kind='barh', stacked=True, figsize=(10, 6), 
                     color=['#56b4e9', '#3492c7', '#f0e442', '#f04442'])

plt.title('BUSCO Completeness Assessment')
plt.xlabel('Percentage (%)')
plt.legend(['Complete (Single)', 'Complete (Duplicated)', 'Fragmented', 'Missing'], 
           bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

# Save and show
plt.savefig(os.path.join(OUT_PATH, 'busco_summary_plot.png'), dpi=300)
plt.show()

- How do the proteomes look?
- what is the average completeness score?
- Do you notice anything particular with some of the proteomes?
- Can you differentiate the technical errors to biological effects?
- Do you think the choice of database (tetrapoda_odb12.2) will have an effect on the BUSCO results? if yes, how? (optional step: try running with a different dataset, what results you get?)

# Filter proteome by completeness
Here we will create a basic filter to select those proteomes that pass a certain threshold of completeness

In [ ]:
# Calculate Total Complete (Single + Duplicated)
df_summary['Total_Complete'] = df_summary['Complete_Single'] + df_summary['Complete_Duplicated']

# Set your strictness threshold
THRESHOLD = 80.0  

# Filter the DataFrame
high_quality = df_summary[df_summary['Total_Complete'] >= THRESHOLD]

print(f"--- Quality Filter (Threshold: {THRESHOLD}%) ---")
print(f"Total proteomes passing: {len(high_quality)} / {len(df_summary)}")
print("Retained species:")
print(high_quality.index.tolist())

- How many of the proteomes pass the threshold, and why?
- Do you see a consistent trend with the proteomes?
- Can you imagine a biological context where you need to be much more inclusive with the threshold?
- If you do not filter, what is the impact of low quality proteomes to the downstream analysis?

# top50 BUSCOs presence/absence heatmap
An additional output of BUSCO is the complete BUSCO genes presence/absence table. As an advanced analysis, you could create a heatmap of selected BUSCOs across all your dataset. This will give you the opportunity to evaluate clade-specific adaptation or gene co-evolution. To obtain such a heatmap you will need to load the full_table.tsv results file of all species in a single dataframe. Then, as an example, you will plot the top 50 BUSCOs arrays.


In [ ]:
# Assuming BUSCO_RES_FOLDER, proteomes, and OUT_PATH are already defined in Step 0
matrix_dict = {}

for species in proteomes:
    table_pattern = os.path.join(BUSCO_RES_FOLDER, species, 'run_tetrapoda_odb12.2', 'full_table.tsv')
    matched_paths = glob.glob(table_pattern)
    
    if not matched_paths:
        matched_paths = glob.glob(os.path.join(BUSCO_RES_FOLDER, species, 'full_table.tsv'))
        
    if matched_paths:
        df_full = pd.read_csv(matched_paths[0], sep='\t', comment='#', header=None, index_col=False,
                              names=['Busco_id', 'Status', 'Sequence', 'Score', 'Length', 'OrthoDB_url', 'Description'])
        
        # Drop duplicate rows based on Busco_id to handle Duplicated genes
        df_dedup = df_full.drop_duplicates(subset=['Busco_id'])
        
        # Store the status indexed by the unique BUSCO ID
        matrix_dict[species] = df_dedup.set_index('Busco_id')['Status']

# Build the master matrix and fill gaps with 'Missing'
df_matrix = pd.DataFrame(matrix_dict).fillna('Missing')

# Map statuses to integers (Handling both 'Single' and 'Complete' naming conventions)
status_map = {'Single': 3, 'Complete': 3, 'Duplicated': 2, 'Fragmented': 1, 'Missing': 0}
df_numeric = df_matrix.replace(status_map)

# Define custom discrete colors (Standard BUSCO palette)
# 0: Missing (Red), 1: Fragmented (Yellow), 2: Duplicated (Dark Blue), 3: Complete (Light Blue)
hex_colors = ['#de2d26', '#fec44f', '#2c7fb8', '#34b1eb']
cmap_discrete = ListedColormap(hex_colors)

# Plot Heatmap (Top 50 Genes)
plt.figure(figsize=(10, 8))

# Disable the default colorbar (cbar=False) and enforce value limits
ax = sns.heatmap(df_numeric.head(50), cmap=cmap_discrete, vmin=0, vmax=3, cbar=False, linewidths=0.5)

# Create a custom discrete legend
legend_labels = ['Missing', 'Fragmented', 'Duplicated', 'Complete']
patches = [mpatches.Patch(color=hex_colors[i], label=legend_labels[i]) for i in range(4)]

# Place the legend outside the heatmap
plt.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left', title="BUSCO Status")

plt.title('BUSCO Presence/Absence Matrix (Top 50 Genes)')
plt.xlabel('Species')
plt.ylabel('BUSCO ID')
plt.tight_layout()

plt.savefig(os.path.join(OUT_PATH, 'busco_presence_heatmap_top50.png'), dpi=300)
plt.show()

- If you spot a column where several closely-related species miss a column, how does it compare to a single species missing a BUSCO gene? How likely it is of being an artifact?
- What if a BUSCO is missing in all the species in the whole dataset? what could this tell us about the selected reference lineage? 

# Extract single copy orthologs
Another important step that can be done after a BUSCO run is to extract the genes that are found as single copy in the entire dataset. Single copy genes are a common input of a few consecutive steps in an evolutionary analysis. For example, we will use them to create a species tree in Module 3.

In [ ]:
# Filter the master matrix for rows where ALL columns equal 'Complete'
# (If you only want to use the high_quality species, change df_matrix to df_matrix[high_quality.index])
core_genes = df_matrix[(df_matrix == 'Complete').all(axis=1)]

print(f"Found {len(core_genes)} universal single-copy orthologs across all analyzed species.")
print("\nSample of Core Gene IDs:")
print(core_genes.index.tolist()[:10])

# Save this list to a file to feed into your downstream alignment/tree tools
core_genes_path = os.path.join(OUT_PATH, 'core_single_copy_buscos.txt')
with open(core_genes_path, 'w') as f:
    for busco_id in core_genes.index:
        f.write(f"{busco_id}\n")

print(f"\nSaved core gene ID list to: {core_genes_path}")

- Why is it so important to select genes that are present strictly in one copy in each species? What would happen if you consider duplicated genes in some species?
- If you start including more and more species in your dataset, you might find that there are no genes shared by all. How would you modify the script so that you can still extract a gene set from such a dataset?